# BrandSphere AI — Week 5: Colour Palette Engine (KMeans)
**CRS AI Capstone 2025–26**

Covers:
1. KMeans clustering on hex colour space
2. Industry × personality colour psychology mapping
3. Palette harmony scoring (HSV analysis)
4. WCAG contrast ratio checker
5. Dark/light mode palette transformation

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn Pillow 2>/dev/null
import subprocess, os, sys
r = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
os.makedirs('assets/sample_exports',exist_ok=True)
print('✅ Ready')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
from sklearn.cluster import KMeans
import colorsys
import warnings; warnings.filterwarnings('ignore')
print('✅ Libraries loaded')

## 1. Colour Space Utilities

In [ ]:
def hex_to_rgb(h: str) -> list:
    h = h.lstrip('#')
    return [int(h[i:i+2],16) for i in (0,2,4)]

def rgb_to_hex(rgb) -> str:
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0]),int(rgb[1]),int(rgb[2]))

def rgb_to_hsv(rgb) -> tuple:
    r,g,b = [v/255 for v in rgb]
    return colorsys.rgb_to_hsv(r,g,b)

def adjust_saturation(hex_color: str, factor: float) -> str:
    r,g,b = [v/255 for v in hex_to_rgb(hex_color)]
    h,s,v = colorsys.rgb_to_hsv(r,g,b)
    s2 = min(1.0, max(0.0, s*factor))
    r2,g2,b2 = colorsys.hsv_to_rgb(h,s2,v)
    return rgb_to_hex([r2*255,g2*255,b2*255])

print('Utilities loaded. Testing:')
print(f'  #1B3A6B → RGB: {hex_to_rgb("#1B3A6B")}')
print(f'  [27,58,107] → Hex: {rgb_to_hex([27,58,107])}')
print(f'  #C9A84C saturation ×1.5: {adjust_saturation("#C9A84C", 1.5)}')

## 2. KMeans Colour Extraction

In production, KMeans extracts dominant colours from logo images. Here we demonstrate on a synthetically generated colour matrix representing an industry's typical brand colours.

In [ ]:
def extract_palette_kmeans(seed_colors: list, k: int = 5) -> list:
    """
    Apply KMeans clustering to a set of colours.
    In production: replace seed_colors with pixels from a logo image.
    """
    # Convert hex seeds to RGB arrays
    rgb_array = np.array([hex_to_rgb(c) for c in seed_colors], dtype=float)
    
    # Generate variations around each seed (simulates image pixel extraction)
    expanded = []
    for color in rgb_array:
        for _ in range(50):
            noise = np.random.normal(0, 15, 3)
            varied = np.clip(color + noise, 0, 255)
            expanded.append(varied)
    expanded = np.array(expanded)
    
    # KMeans clustering
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(expanded)
    
    centroids = kmeans.cluster_centers_
    labels    = kmeans.labels_
    
    # Sort by cluster size (most dominant first)
    counts  = np.bincount(labels)
    sorted_idx = np.argsort(counts)[::-1]
    
    palette = []
    names   = ['Primary','Secondary','Accent','Background','Text / CTA']
    for i, idx in enumerate(sorted_idx[:k]):
        palette.append({
            'name': names[i],
            'hex':  rgb_to_hex(centroids[idx]),
            'rgb':  centroids[idx].astype(int).tolist(),
            'coverage': round(counts[idx]/len(labels)*100, 1)
        })
    return palette

# Test on Technology / Software brand colours
tech_seeds = ['#1B3A6B','#2E86AB','#4CC9F0','#F0F4F8','#1A1A2E']
tech_palette = extract_palette_kmeans(tech_seeds, k=5)

print('KMeans palette — Technology / Software (Minimalist):')
for p in tech_palette:
    print(f"  {p['name']:15s}: {p['hex']}  RGB={p['rgb']}  coverage={p['coverage']}%")

## 3. Industry Palette Mapping

In [ ]:
from src.config import COLOR_PSYCHOLOGY

print(f'Industries mapped: {len(COLOR_PSYCHOLOGY)}')
print()

# Extract and display all industry palettes
for industry, colors in COLOR_PSYCHOLOGY.items():
    print(f'{industry}:')
    for role, hex_val in colors.items():
        rgb = hex_to_rgb(hex_val)
        print(f'  {role:10s}: {hex_val}  RGB={rgb}')
    print()

## 4. Palette Harmony Scoring

In [ ]:
def score_palette_harmony(palette: list) -> float:
    """
    Score palette harmony using HSV analysis.
    Checks: hue spread, saturation consistency, value contrast.
    Returns 0.0 - 1.0
    """
    if len(palette) < 2:
        return 0.5
    
    hsvs = [rgb_to_hsv(p['rgb']) for p in palette]
    hues  = [h[0] for h in hsvs]
    sats  = [h[1] for h in hsvs]
    vals  = [h[2] for h in hsvs]
    
    # Hue spread: good palettes use 2-4 distinct hue regions
    hue_spread = np.std(hues)
    hue_score  = min(1.0, hue_spread * 5)
    
    # Saturation consistency: similar saturation = cohesive
    sat_score = 1.0 - min(1.0, np.std(sats) * 3)
    
    # Value contrast: need at least one dark and one light
    val_range  = max(vals) - min(vals)
    val_score  = min(1.0, val_range * 2)
    
    harmony = (hue_score * 0.4 + sat_score * 0.3 + val_score * 0.3)
    return round(harmony, 3)

# Score all industry palettes
print('Palette harmony scores by industry:')
print('-' * 50)
for industry, colors in COLOR_PSYCHOLOGY.items():
    palette = [{'name':k,'hex':v,'rgb':hex_to_rgb(v)} for k,v in colors.items()]
    score   = score_palette_harmony(palette)
    bar     = '█' * int(score * 20)
    print(f'{industry[:35]:35s}: {score:.3f} {bar}')

## 5. WCAG Contrast Ratio Checker

In [ ]:
def relative_luminance(rgb):
    """WCAG 2.1 relative luminance formula."""
    def linearize(c):
        c = c / 255
        return c/12.92 if c <= 0.04045 else ((c+0.055)/1.055)**2.4
    r,g,b = rgb
    return 0.2126*linearize(r) + 0.7152*linearize(g) + 0.0722*linearize(b)

def contrast_ratio(hex1: str, hex2: str) -> float:
    l1 = relative_luminance(hex_to_rgb(hex1))
    l2 = relative_luminance(hex_to_rgb(hex2))
    lighter, darker = max(l1,l2), min(l1,l2)
    return round((lighter+0.05)/(darker+0.05), 2)

def wcag_rating(ratio: float) -> str:
    if ratio >= 7.0:   return 'AAA ✅'
    elif ratio >= 4.5: return 'AA  ✅'
    elif ratio >= 3.0: return 'AA Large ⚠️'
    else:              return 'Fail ❌'

# Test all industry palettes
print('WCAG Contrast Ratios — Primary on Background')
print('-' * 55)
for industry, colors in list(COLOR_PSYCHOLOGY.items())[:6]:
    ratio  = contrast_ratio(colors['primary'], colors['neutral'])
    rating = wcag_rating(ratio)
    print(f'{industry[:35]:35s}: {ratio:5.2f}:1  {rating}')

## 6. Visualise Industry Palettes

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 12))
fig.suptitle('BrandSphere AI — Industry Colour Palettes', color='#C9A84C', fontsize=14, y=1.01)

industries = list(COLOR_PSYCHOLOGY.keys())
for ax, industry in zip(axes.flatten(), industries):
    colors_dict = COLOR_PSYCHOLOGY[industry]
    hex_colors  = list(colors_dict.values())
    names       = list(colors_dict.keys())
    
    for j, (hx, nm) in enumerate(zip(hex_colors, names)):
        rect = patches.Rectangle((j*0.2, 0), 0.2, 1, facecolor=hx)
        ax.add_patch(rect)
        rgb = hex_to_rgb(hx)
        lum = 0.299*rgb[0] + 0.587*rgb[1] + 0.114*rgb[2]
        txt_color = 'white' if lum < 128 else 'black'
        ax.text(j*0.2+0.1, 0.5, nm[:4], ha='center', va='center',
                fontsize=7, color=txt_color, rotation=90)
    
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.set_title(industry.split('/')[0].strip(), color='white', fontsize=9, pad=6)

plt.tight_layout()
plt.savefig('assets/sample_exports/04_palettes.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print('✅ Palette visualisation saved')

## Summary

| Method | Input | Output |
|---|---|---|
| KMeans (k=5) | Logo image pixels / seed colours | 5 dominant brand colours |
| Harmony scoring | HSV decomposition | 0.0–1.0 score |
| WCAG checker | Two hex colours | Contrast ratio + AA/AAA rating |
| Industry mapping | Industry + personality | Pre-computed palette from COLOR_PSYCHOLOGY table |